# Analisis Multivariante

## Objetivo del analisis

En este notebook se analiza de forma conjunta la relacion entre salario, empleabilidad, cualificacion academica y experiencia profesional/tecnica.

Las hipotesis trabajadas son:

- **H1:** Cuanta mayor edad o trayectoria profesional, mayor salario.
- **H2:** A mayor cualificacion academica, mayor empleabilidad.
- **H3:** A mayor experiencia profesional y mayor experiencia tecnica, mayor salario.

En el dataset no existe una columna que mida directamente la cantidad de lenguajes conocidos. Por este motivo, se utiliza `YearsCode` como aproximacion a experiencia tecnica, ya que representa los anos totales programando.

## Pasos seguidos

0. Establecer hipotesis multivariantes.
1. Cargar el dataset limpio.
2. Preparar variables derivadas para salario, empleabilidad, educacion y experiencia.
3. Analizar salario y empleabilidad por nivel educativo.
4. Analizar salario por experiencia profesional y experiencia tecnica.
5. Cruzar educacion y experiencia para estudiar efectos combinados.
6. Calcular correlaciones entre variables principales.
7. Construir modelos sencillos para estimar salario y empleabilidad.
8. Interpretar resultados y extraer conclusiones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="Set2")

# Cargar dataset limpio de forma robusta tanto desde el notebook como desde la raiz del repo
possible_paths = [
    Path("../data/develop_dataset_2025_limpio.csv"),
    Path("src/data/develop_dataset_2025_limpio.csv")
]

data_path = next(path for path in possible_paths if path.exists())
df = pd.read_csv(data_path, index_col=0)

df.head()

In [ ]:
# Corregir anomalia detectada en analisis anteriores
# YearsCode = 100 no es coherente con el rango de edad declarado en esos registros.
df.loc[df["YearsCode"] == 100, "YearsCode"] = pd.NA

# Variables principales
salary_col = "ConvertedCompYearly"
education_col = "EdLevel"
employment_col = "Employment"
workexp_col = "WorkExp"
yearscode_col = "YearsCode"

df_multi = df.copy()
df_multi.shape

In [ ]:
# Crear variables derivadas para el analisis multivariante

def score_ed_level(value):
    if pd.isna(value):
        return np.nan
    if value.startswith("Primary"):
        return 1
    if value.startswith("Secondary"):
        return 2
    if value.startswith("Some college"):
        return 3
    if value.startswith("Associate"):
        return 4
    if value.startswith("Bachelor"):
        return 5
    if value.startswith("Master"):
        return 6
    if value.startswith("Professional"):
        return 7
    return np.nan

experience_bins = [0, 5, 10, 15, 20, 30, 60]
experience_labels = ["0-5", "6-10", "11-15", "16-20", "21-30", "31+"]

df_multi["education_score"] = df_multi[education_col].apply(score_ed_level)
df_multi["is_employed"] = df_multi[employment_col].eq("Employed").astype(int)
df_multi["log_salary"] = np.log10(df_multi[salary_col])
df_multi["WorkExp_group"] = pd.cut(
    df_multi[workexp_col],
    bins=experience_bins,
    labels=experience_labels,
    include_lowest=True
)

df_multi[[salary_col, education_col, employment_col, workexp_col, yearscode_col, "education_score", "is_employed", "WorkExp_group"]].head()

## H2: cualificacion academica, empleabilidad y salario

Primero se analiza la hipotesis H2 porque en los analisis anteriores se habia tratado principalmente como relacion entre nivel educativo y empleabilidad, aunque tambien se reviso el salario como indicador adicional.

In [ ]:
# Porcentaje de empleados por nivel educativo
empleados_por_educacion = (
    df_multi.groupby(education_col)["is_employed"]
    .agg(["count", "mean"])
    .assign(porcentaje_empleados=lambda x: x["mean"] * 100)
    .sort_values("porcentaje_empleados", ascending=False)
    .round(2)
)

empleados_por_educacion

In [ ]:
# Salario por nivel educativo
salario_por_educacion = (
    df_multi.groupby(education_col)[salary_col]
    .agg(["count", "mean", "median"])
    .sort_values("median", ascending=False)
    .round(2)
)

salario_por_educacion

In [ ]:
# Visualizar salario por nivel educativo
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_multi, x=salary_col, y=education_col)
plt.title("Distribucion salarial por nivel educativo")
plt.xlabel("Salario anual")
plt.ylabel("Nivel educativo")
plt.xscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar empleabilidad por nivel educativo
empleados_plot = empleados_por_educacion.reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=empleados_plot, x="porcentaje_empleados", y=education_col)
plt.title("Porcentaje de personas empleadas por nivel educativo")
plt.xlabel("Porcentaje de empleados")
plt.ylabel("Nivel educativo")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

### Conclusion H2

Los datos respaldan parcialmente la H2. La empleabilidad es mayor en los niveles educativos altos: master alcanza un 87.55%, grado universitario un 85.38% y titulo profesional o doctorado un 85.09% de personas empleadas. En cambio, primaria se queda en un 44.44%.

El salario tambien mejora en los niveles de mayor cualificacion. La mediana salarial mas alta aparece en `Professional degree`, con 109,438 USD anuales, seguida por master y grado universitario. Por tanto, en este dataset la cualificacion academica se relaciona tanto con mayor empleabilidad como con mejores salarios, aunque el indicador principal de H2 sigue siendo la empleabilidad.

## H3: experiencia profesional, experiencia tecnica y salario

A continuacion se analiza si el salario aumenta al combinar experiencia laboral (`WorkExp`) y experiencia tecnica (`YearsCode`).

In [ ]:
# Salario por rangos de experiencia laboral
salario_por_experiencia = (
    df_multi.groupby("WorkExp_group", observed=True)[salary_col]
    .agg(["count", "mean", "median"])
    .round(2)
)

salario_por_experiencia

In [ ]:
# Visualizar salario por rangos de experiencia laboral
plt.figure(figsize=(8, 5))
sns.barplot(data=df_multi, x="WorkExp_group", y=salary_col, estimator="median")
plt.title("Mediana salarial por rango de experiencia laboral")
plt.xlabel("Rango de experiencia laboral")
plt.ylabel("Mediana salarial")
plt.tight_layout()
plt.show()

In [ ]:
# Relacion conjunta entre experiencia laboral, experiencia tecnica y salario
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_multi,
    x=workexp_col,
    y=yearscode_col,
    size=salary_col,
    hue="education_score",
    sizes=(20, 300),
    alpha=0.55,
    palette="viridis"
)
plt.title("Experiencia laboral, experiencia tecnica y salario")
plt.xlabel("Anios de experiencia laboral")
plt.ylabel("Anios programando")
plt.legend(title="Nivel educativo", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Conclusion H3

La experiencia muestra una relacion positiva con el salario, especialmente al agrupar por rangos. La mediana salarial pasa de 46,496 USD en perfiles de 0-5 anos de experiencia a 127,616 USD en perfiles con 31 anos o mas.

La experiencia tecnica (`YearsCode`) tambien se asocia con salarios mas altos, y en el analisis multivariante aparece como una de las variables con mayor relacion con el salario. La H3 se acepta parcialmente: la experiencia y los anos programando ayudan a explicar salarios mas altos, aunque no son los unicos factores.

## Relacion conjunta: educacion + experiencia + salario

Para estudiar la relacion multivariante, se cruzan nivel educativo y experiencia laboral. Este cruce permite ver si el salario depende solo de una variable o de la combinacion entre formacion y trayectoria profesional.

In [ ]:
# Mediana salarial cruzando nivel educativo y experiencia laboral
salary_edu_exp = df_multi.pivot_table(
    values=salary_col,
    index=education_col,
    columns="WorkExp_group",
    aggfunc="median",
    observed=True
).round(2)

salary_edu_exp

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(salary_edu_exp, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Mediana salarial por nivel educativo y experiencia laboral")
plt.xlabel("Rango de experiencia laboral")
plt.ylabel("Nivel educativo")
plt.tight_layout()
plt.show()

In [ ]:
# Empleabilidad cruzando nivel educativo y experiencia laboral
employment_edu_exp = df_multi.pivot_table(
    values="is_employed",
    index=education_col,
    columns="WorkExp_group",
    aggfunc="mean",
    observed=True
).mul(100).round(2)

employment_edu_exp

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(employment_edu_exp, annot=True, fmt=".1f", cmap="Greens")
plt.title("Porcentaje de empleados por nivel educativo y experiencia laboral")
plt.xlabel("Rango de experiencia laboral")
plt.ylabel("Nivel educativo")
plt.tight_layout()
plt.show()

### Interpretacion conjunta

El cruce entre educacion y experiencia muestra que el salario no depende de una sola variable. Los niveles educativos altos suelen presentar mejores medianas, pero la experiencia laboral tambien desplaza el salario hacia arriba dentro de varios niveles academicos.

En empleabilidad, los niveles de master, grado universitario y titulo profesional mantienen porcentajes altos, aunque los grupos con mucha experiencia pueden incluir perfiles retirados, autonomos o no empleados, lo que reduce el porcentaje de `Employed` en algunos tramos.

## Correlaciones entre variables principales

Se usa `log_salary` para reducir el efecto de los salarios extremos. Esto permite interpretar mejor la relacion entre salario, experiencia, educacion y empleabilidad.

In [ ]:
corr_vars = ["log_salary", workexp_col, yearscode_col, "education_score", "is_employed"]
corr_matrix = df_multi[corr_vars].corr(numeric_only=True).round(3)

corr_matrix

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Matriz de correlacion multivariante")
plt.tight_layout()
plt.show()

### Interpretacion de correlaciones

La relacion mas clara con el salario logaritmico aparece en `YearsCode` (0.316), seguida por `education_score` (0.248) y `WorkExp` (0.220). La variable `is_employed` tambien se relaciona positivamente con el salario logaritmico (0.167), aunque de forma mas moderada.

Esto refuerza la idea de que el salario se explica mejor combinando experiencia tecnica, experiencia laboral, educacion y situacion laboral.

## Modelo multivariante para salario

Se construye un modelo lineal sencillo con variables estandarizadas. El objetivo no es predecir perfectamente el salario, sino comparar que variables tienen mayor peso relativo al analizarse juntas.

In [ ]:
# Modelo lineal sencillo para salario logaritmico
model_salary = df_multi[["log_salary", workexp_col, yearscode_col, "education_score", "is_employed"]].dropna()

y = model_salary["log_salary"].values
X = model_salary[[workexp_col, yearscode_col, "education_score", "is_employed"]].copy()

# Estandarizar variables para comparar coeficientes
X_std = (X - X.mean()) / X.std(ddof=0)
X_matrix = np.column_stack([np.ones(len(X_std)), X_std.values])

coef = np.linalg.lstsq(X_matrix, y, rcond=None)[0]
pred = X_matrix @ coef
r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()

coef_salary = pd.DataFrame({
    "variable": ["intercept"] + list(X.columns),
    "coeficiente": coef
}).round(4)

print(f"R2 del modelo: {r2:.3f}")
print(f"Registros usados: {len(model_salary)}")
coef_salary

### Conclusion del modelo de salario

El modelo explica aproximadamente un 16.3% de la variabilidad del salario logaritmico. No es un modelo predictivo fuerte, pero sirve para comparar efectos relativos.

Al analizar las variables conjuntamente, `YearsCode` mantiene el mayor peso positivo (0.1883), seguido por `education_score` (0.1364) e `is_employed` (0.1047). `WorkExp` tambien es positivo, pero su peso baja (0.0350) cuando se controla por anos programando y educacion.

Esto sugiere que la experiencia tecnica y la cualificacion academica ayudan a explicar el salario, mientras que la experiencia laboral general aporta informacion adicional pero parcialmente solapada con `YearsCode`.

## Modelo multivariante para empleabilidad

Para H2 se analiza tambien la empleabilidad como variable objetivo. Se usa un modelo lineal sencillo sobre `is_employed`, interpretado solo de forma exploratoria.

In [ ]:
# Modelo exploratorio para empleabilidad
model_employment = df_multi[["is_employed", workexp_col, yearscode_col, "education_score"]].dropna()

y_emp = model_employment["is_employed"].values
X_emp = model_employment[[workexp_col, yearscode_col, "education_score"]].copy()

X_emp_std = (X_emp - X_emp.mean()) / X_emp.std(ddof=0)
X_emp_matrix = np.column_stack([np.ones(len(X_emp_std)), X_emp_std.values])

coef_emp = np.linalg.lstsq(X_emp_matrix, y_emp, rcond=None)[0]
pred_emp = X_emp_matrix @ coef_emp
r2_emp = 1 - ((y_emp - pred_emp) ** 2).sum() / ((y_emp - y_emp.mean()) ** 2).sum()

coef_employment = pd.DataFrame({
    "variable": ["intercept"] + list(X_emp.columns),
    "coeficiente": coef_emp
}).round(4)

print(f"R2 del modelo: {r2_emp:.3f}")
print(f"Registros usados: {len(model_employment)}")
coef_employment

### Conclusion del modelo de empleabilidad

El modelo de empleabilidad tiene una capacidad explicativa baja, con un R2 aproximado de 4.8%. Esto indica que la situacion laboral no se explica completamente solo con educacion y experiencia.

Aun asi, `education_score` mantiene una relacion positiva con estar empleado (0.0407), lo que refuerza parcialmente la H2. `YearsCode` tambien aparece con efecto positivo (0.0529). En cambio, `WorkExp` aparece con coeficiente negativo (-0.0983), probablemente porque los tramos de mayor experiencia incluyen mas perfiles retirados, autonomos o no clasificados como `Employed`.

Por tanto, la educacion se relaciona con la empleabilidad, pero no basta para explicarla por completo.

## Conclusion final del analisis multivariante

El analisis multivariante muestra que salario, empleabilidad, cualificacion academica y experiencia estan relacionados, pero no de forma aislada.

La H2 se acepta parcialmente porque los niveles educativos mas altos presentan mayores porcentajes de personas empleadas y mejores medianas salariales. Sin embargo, la empleabilidad tambien depende de otros factores no incluidos directamente en estas hipotesis, como pais, tipo de rol, industria o situacion profesional concreta.

La H3 tambien se acepta parcialmente. La experiencia laboral y, sobre todo, la experiencia tecnica medida con `YearsCode`, se asocian con salarios mas altos. Aun asi, la dispersion salarial sigue siendo elevada, por lo que la experiencia no explica por si sola las diferencias de salario.

En conjunto, el salario parece estar mas relacionado con la combinacion de experiencia tecnica, cualificacion academica, experiencia laboral y situacion laboral. La empleabilidad, por su parte, mejora con la cualificacion academica, aunque el modelo muestra que hay otros factores relevantes que tambien deberian considerarse.